# DC4 lightcurve comparison: model / FITS ratios

Run this notebook independently of the catalog generator. It reads only the small
`source_catalog_DC4_lightcurve_comparison.json` snapshot and needs NumPy and
Matplotlib—not cosipy, event FITS, orientation, catalog YAML, or response cubes.

**First use:** in the generator, run the new “Save a lightweight comparison cache”
cells after its comparison calculation. If that completed calculation is still
in memory, export it without rerunning it. Saved notebook figures/tables alone do
not preserve the per-energy-bin arrays needed here. Afterward, this plotting
notebook can be run in a fresh kernel as often as needed.

The cache records the generator inputs and timestamp; it does not refresh itself
when YAML or model parameters change. Undefined ratios (zero FITS denominator)
are omitted, never replaced by zero or one. All plots show ratios only.


In [ ]:
# Lightweight plotting code: no cosipy, FITS, catalog models, or responses.
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

COMPARISON_METHODS = [
    ("Parallel chunks + LC response", "tab:blue", "s", "-"),
    ("No chunks + LC response", "tab:orange", "^", "--"),
    ("Parallel chunks, duration only", "tab:green", "x", ":"),
]


def validate_comparison_snapshot(snapshot):
    if snapshot.get("schema") != "dc4-lightcurve-count-comparison-v1":
        raise ValueError("Unsupported comparison-cache schema")
    edges = np.asarray(snapshot["energy_edges_keV"], dtype=float)
    if (edges.ndim != 1 or edges.size < 2 or not np.all(np.isfinite(edges))
            or np.any(edges <= 0) or np.any(np.diff(edges) <= 0)):
        raise ValueError("Energy edges must be finite, positive and increasing")
    labels = {"FITS", *(method[0] for method in COMPARISON_METHODS)}
    seen = set()
    sources_by_selection = {}
    for record in snapshot["records"]:
        key = (record["selection"], record["source"])
        if key in seen:
            raise ValueError(f"Duplicate comparison record: {key}")
        seen.add(key)
        sources_by_selection.setdefault(key[0], set()).add(key[1])
        if set(record["counts"]) != labels:
            raise ValueError(f"Missing/unknown comparison methods: {key}")
        for values in record["counts"].values():
            values = np.asarray(values, dtype=float)
            if (values.shape != (edges.size - 1,) or not np.all(np.isfinite(values))
                    or np.any(values < 0)):
                raise ValueError(f"Invalid count spectrum: {key}")
    if not seen:
        raise ValueError("Comparison cache has no records")
    if len(snapshot["variable_sources"]) != len(set(snapshot["variable_sources"])):
        raise ValueError("Duplicate variable-source names")
    for selection, sources in sources_by_selection.items():
        if not set(snapshot["variable_sources"]).issubset(sources):
            raise ValueError(f"Missing variable-source results for {selection}")
    return snapshot


def model_fits_ratio(model, observed):
    """A zero FITS denominator is undefined, not zero or unity agreement."""
    model, observed = np.broadcast_arrays(np.asarray(model, float), np.asarray(observed, float))
    return np.divide(model, observed, out=np.full(model.shape, np.nan), where=observed > 0)


def ratio_axis(ax, scale):
    if scale not in {"linear", "symlog"}:
        raise ValueError("Use 'linear' or 'symlog'; both preserve zero model predictions")
    if scale == "symlog":
        ax.set_yscale("symlog", linthresh=.1)
    ax.axhline(1., color="black", ls="--", lw=1)
    ax.set_ylabel("Model / FITS")
    ax.set_facecolor("white")
    ax.grid(axis="y", which="major", alpha=.2)


def plot_total_ratios(snapshot, scale="symlog"):
    """Ratio of summed predicted counts to summed native FITS counts per source."""
    validate_comparison_snapshot(snapshot)
    records = {(r["selection"], r["source"]): r["counts"] for r in snapshot["records"]}
    selections = list(dict.fromkeys(r["selection"] for r in snapshot["records"]))
    figures = []
    for selection in selections:
        sources = snapshot["variable_sources"]
        x = np.arange(len(sources))
        observed = np.array([np.sum(records[selection, key]["FITS"]) for key in sources])
        fig, ax = plt.subplots(figsize=(18, 5), constrained_layout=True)
        fig.patch.set_facecolor("white")
        for method, color, marker, _ in COMPARISON_METHODS:
            modeled = np.array([np.sum(records[selection, key][method]) for key in sources])
            ax.plot(x, model_fits_ratio(modeled, observed), marker=marker,
                    ls="none", color=color, label=method)
        labels = [key + (" *" if obs == 0 else "") for key, obs in zip(sources, observed)]
        ax.set_xticks(x, labels, rotation=90)
        ratio_axis(ax, scale)
        ax.set_title(f"{selection}: total model / FITS ratios")
        if np.any(observed == 0):
            ax.text(.01, .98, "* No FITS events: ratio undefined (not plotted)",
                    transform=ax.transAxes, va="top", fontsize=9)
        ax.legend(frameon=False)
        figures.append(fig)
    return figures


def plot_energy_ratios(snapshot, sources, scale="symlog"):
    """Measured-energy-bin count ratios, not incident spectral flux ratios."""
    validate_comparison_snapshot(snapshot)
    if not sources or len(sources) != len(set(sources)):
        raise ValueError("Choose at least one source, without duplicates")
    edges = np.asarray(snapshot["energy_edges_keV"], dtype=float) / 1000.
    records = {(r["selection"], r["source"]): r["counts"] for r in snapshot["records"]}
    selections = list(dict.fromkeys(r["selection"] for r in snapshot["records"]))
    missing = [(selection, key) for selection in selections for key in sources
               if (selection, key) not in records]
    if missing:
        raise ValueError(f"Missing source/selection results: {missing}")
    fig, axes = plt.subplots(len(sources), len(selections),
                             figsize=(13, 3.5 * len(sources)), squeeze=False,
                             constrained_layout=True)
    fig.patch.set_facecolor("white")
    for row, key in enumerate(sources):
        for col, selection in enumerate(selections):
            ax = axes[row, col]
            spectra = records[selection, key]
            observed = np.asarray(spectra["FITS"], dtype=float)
            for method, color, _, linestyle in COMPARISON_METHODS:
                ax.stairs(model_fits_ratio(spectra[method], observed), edges,
                          color=color, ls=linestyle, label=method, baseline=None)
            ratio_axis(ax, scale)
            ax.set_xscale("log")
            ax.set_xlim(edges[0], edges[-1])
            ax.set_title(f"{key} — {selection}")
            ax.set_xlabel("Measured energy (MeV)")
            if not np.any(observed > 0):
                ax.text(.5, .5, "No FITS events in this selection\nRatios undefined",
                        transform=ax.transAxes, ha="center", va="center")
                ax.set_ylim(0., 2.)
            elif np.any(observed == 0):
                ax.text(.02, .98, "Empty FITS bins omitted", transform=ax.transAxes,
                        va="top", fontsize=8)
            if row == 0 and col == 0:
                ax.legend(fontsize=9, frameon=False)
    return fig


## Load the saved comparison snapshot


In [ ]:
# Set an explicit path if you keep the snapshot elsewhere.
cache_name = "source_catalog_DC4_lightcurve_comparison.json"
cache_candidates = [Path.cwd() / cache_name]
for parent in (Path.cwd(), *Path.cwd().parents):
    cache_candidates.append(parent / "docs/tutorials/spectral_fits/continuum_fit/AGN" / cache_name)
COMPARISON_CACHE_PATH = next((p for p in cache_candidates if p.is_file()), cache_candidates[0])
if not COMPARISON_CACHE_PATH.is_file():
    raise FileNotFoundError(
        f"Missing comparison snapshot: {COMPARISON_CACHE_PATH}. "
        "Run the generator's 'Save a lightweight comparison cache' cells first "
        "(using its existing results if still in memory), or set COMPARISON_CACHE_PATH."
    )
comparison_snapshot = validate_comparison_snapshot(json.loads(COMPARISON_CACHE_PATH.read_text()))
print("Snapshot:", COMPARISON_CACHE_PATH)
print("Created UTC:", comparison_snapshot["created_utc"])
print("Generator provenance:", json.dumps(comparison_snapshot["provenance"], indent=2))


## Total model / FITS ratios by variable source


In [ ]:
# Use "linear" for a closer view around unity; symlog also displays zero ratios.
RATIO_SCALE = "symlog"
total_ratio_figures = plot_total_ratios(comparison_snapshot, scale=RATIO_SCALE)
plt.show()


## Energy-binned model / FITS ratios for selected sources


In [ ]:
PLOT_SOURCES = ["maxi_j1348", "co_nova_continuum", "grb_bn090424592", "mgf_051103"]
energy_ratio_figure = plot_energy_ratios(comparison_snapshot, PLOT_SOURCES, scale=RATIO_SCALE)
plt.show()
